# Notebook 01f-gold — Gold-Standard Validation (exact SVD vs. power iteration)

## Purpose

Standalone companion to `01e_jacobian_topk_probe.ipynb`'s former Cell 8b. Split out so it
can run in its **own fresh kernel** after Cell 8's 32-layer power-iteration sweep
completes there — jacrev's chunked exact-Jacobian computation is the single heaviest
allocation in the whole pipeline, and a clean process avoids VRAM fragmentation left over
from that sweep (rather than relying on `gc.collect()` / `torch.cuda.empty_cache()` alone
to undo it).

This notebook does **not** recompute `mean_h_by_layer` or `context_tokens` — it loads them
from disk, where Cell 8 of `01e_jacobian_topk_probe.ipynb` saves them
(`jacobian_outputs/violence/mean_h_by_layer.pt`, `context_tokens.pt`). Run that notebook's
Cell 8 first so those two files exist, then run this notebook end-to-end in a fresh kernel.

## Method

Compute the **exact** `d_model x d_model` Jacobian at a single layer (`L_STAR`, the layer
01c's DIM search selected for `violence`) via chunked reverse-mode autodiff
(`torch.func.jacrev(..., chunk_size=...)`), take its exact SVD, and compare `sigma_1` /
`v_1` against `jacobian_topk`'s power-iteration estimate at the same `h0`. Passing this is
what makes the Cell 9/10 plots in the main notebook trustworthy.

**Do not loop this over all 32 layers.** Materializing the full Jacobian — even chunked —
means running a batch of `d_model` (4096) "virtual copies" through the remaining decoder
layers; that is exactly the cost power iteration exists to avoid, and is only affordable
here as a one-time ground-truth check at the layer that matters most.

## Cell 1 — Imports and Output Directory Setup

In [1]:
import gc
import json
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from typing import Callable, Tuple

torch.manual_seed(42)

OUTPUT_ROOT = 'jacobian_outputs'
CATEGORY    = 'violence'   # must match the run of 01e_jacobian_topk_probe.ipynb that produced
                            # mean_h_by_layer.pt / context_tokens.pt

os.makedirs(os.path.join(OUTPUT_ROOT, CATEGORY), exist_ok=True)

print('Imports complete (TransformerLens loaded).')
print(f'Output directory: {OUTPUT_ROOT}/{CATEGORY}/')

Imports complete (TransformerLens loaded).
Output directory: jacobian_outputs/violence/


## Cell 2 — Configuration

**Update `MODEL_PATH`** to point to your local checkpoint before running anything else
(same value as `01e_jacobian_topk_probe.ipynb` Cell 2).

In [2]:
MODEL_PATH = '/home/samuel/research/llmattacks/llm-attacks/DIR/Llama-3.1-8B-Instruct'
DEVICE     = 'cuda:0'

# Must match T_POS in 01e_jacobian_topk_probe.ipynb -- it is baked into the saved
# context_tokens.pt / mean_h_by_layer.pt only implicitly (they were computed under this
# convention), so keep it consistent by hand.
T_POS = -1

# Power-iteration settings for jacobian_topk (must match the main notebook for a fair
# like-for-like comparison against its power-iteration results).
JAC_NITER = 15
JAC_SEED  = 0

# L_STAR = the layer that actually matters for the reported RSR numbers: the layer
# 01c's DIM search selected for `violence` (dim_outputs/violence/direction_metadata.json
# -> "layer": 16), not an arbitrary midpoint. Only probed once -- looping the full
# Jacobian over all 32 layers would reintroduce the exact cost problem power iteration
# was built to avoid.
#
# JACREV_CHUNK_SIZE starts small (8) because chunked jacrev still runs `chunk_size`
# *simultaneous* copies of the entire remaining forward+backward pass (16 decoder layers,
# d_mlp=14336) via vmap -- on an 8B model that is far heavier than the model weights
# themselves. The computation cell below doubles this automatically on OOM until it fits,
# so raise the starting point only if you know you have VRAM to spare.
L_STAR            = 16
JACREV_CHUNK_SIZE = 8

print(f'MODEL_PATH : {MODEL_PATH}')
print(f'CATEGORY   : {CATEGORY}')
print(f'T_POS      : {T_POS}')
print(f'L_STAR     : {L_STAR}')

MODEL_PATH : /home/samuel/research/llmattacks/llm-attacks/DIR/Llama-3.1-8B-Instruct
CATEGORY   : violence
T_POS      : -1
L_STAR     : 16


## Cell 3 — Load Model and Tokenizer

Unchanged from Notebook 01e / 01c.

In [3]:
print('Loading HuggingFace model ...')
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,   # TransformerLens requires float32 at init
    device_map='cpu',
)
hf_model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Wrapping with TransformerLens HookedTransformer ...')
model = HookedTransformer.from_pretrained(
    'meta-llama/Meta-Llama-3-8B-Instruct',
    hf_model=hf_model,
    tokenizer=tokenizer,
    dtype=torch.float16,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    move_to_device=True,
    device=DEVICE,
)
model.eval()

del hf_model
gc.collect()

D_MODEL    = model.cfg.d_model
NUM_LAYERS = model.cfg.n_layers
FINAL_RESID_KEY = f'blocks.{NUM_LAYERS - 1}.hook_resid_post'

print(f'TransformerLens model ready on {DEVICE}')
print(f'd_model    : {D_MODEL}')
print(f'num_layers : {NUM_LAYERS}')

Loading HuggingFace model ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Wrapping with TransformerLens HookedTransformer ...
Loaded pretrained model meta-llama/Meta-Llama-3-8B-Instruct into HookedTransformer
TransformerLens model ready on cuda:0
d_model    : 4096
num_layers : 32


## Cell 4 — Load `mean_h_by_layer` and `context_tokens` (saved by Cell 8 of the main notebook)

These are the two fixed inputs `F_star` and `h0_star` are built from. Loading them instead
of recomputing means this notebook does not need `data/saladbench_splits/...` or the
16-prompt forward-pass sweep at all -- just the two saved tensors.

In [4]:
mean_h_path = os.path.join(OUTPUT_ROOT, CATEGORY, 'mean_h_by_layer.pt')
context_tokens_path = os.path.join(OUTPUT_ROOT, CATEGORY, 'context_tokens.pt')

assert os.path.exists(mean_h_path), (
    f'{mean_h_path} not found -- run Cell 8 of 01e_jacobian_topk_probe.ipynb first.'
)
assert os.path.exists(context_tokens_path), (
    f'{context_tokens_path} not found -- run Cell 8 of 01e_jacobian_topk_probe.ipynb first.'
)

mean_h_by_layer = torch.load(mean_h_path)               # (NUM_LAYERS, D_MODEL), CPU
context_tokens  = torch.load(context_tokens_path).to(DEVICE)

assert mean_h_by_layer.shape == (NUM_LAYERS, D_MODEL), (
    f'mean_h_by_layer shape {tuple(mean_h_by_layer.shape)} does not match this model '
    f'({NUM_LAYERS}, {D_MODEL}) -- was it saved by a different model/category?'
)

print(f'Loaded mean_h_by_layer: {tuple(mean_h_by_layer.shape)}')
print(f'Loaded context_tokens : {tuple(context_tokens.shape)}')

Loaded mean_h_by_layer: (32, 4096)
Loaded context_tokens : (1, 47)


## Cell 5 — `run_from_layer` (fast variant), `jacobian_topk`, `jacobian_full_chunked`

`jacobian_topk` is copied unchanged from `01e_jacobian_topk_probe.ipynb` (Cell 6), kept
identical so the power-iteration estimate computed below is directly comparable to the
main notebook's.

`make_run_from_layer` is **not** unchanged this time. The original (Cell 5 in the main
notebook) rebuilds `F(h)` by re-running the model from the embeddings through all 32
layers on every call, hooking a substitution in at `blocks.{layer_l}`. That's fine for
`jacobian_topk` (sequential single JVP/VJP calls), but `jacobian_full_chunked` traces `F`
through `torch.func.jacrev(..., chunk_size=...)`, which vmaps `chunk_size` virtual copies
of the *entire remaining forward+backward graph* at once -- including the redundant
0..`layer_l - 1` blocks, which don't depend on `h` at all. That redundant tracing is what
OOM'd even at `chunk_size=1` (see the notebook's earlier run).

The fix: precompute the fixed residual stream at `blocks.{layer_l}.hook_resid_post` once
(`precompute_resid_at_layer`, no grad), then use TransformerLens's `start_at_layer` to
resume the forward pass directly from `layer_l` on every call to `F(h)` -- so `jacrev`
only ever traces blocks `layer_l..31`, not the full model. Mathematically this computes
the exact same function as the hook-based version; it's just cheaper to differentiate.

In [5]:
def precompute_resid_at_layer(model: HookedTransformer, layer_l: int,
                               base_tokens: torch.Tensor) -> torch.Tensor:
    """Run base_tokens through blocks 0..layer_l once (no grad) and return the full
    (batch, seq, d_model) residual stream at blocks.{layer_l}.hook_resid_post.

    This is the fixed input make_run_from_layer resumes from via start_at_layer, instead
    of recomputing embeddings-through-layer_l on every call to F(h).
    """
    with torch.no_grad():
        _, cache = model.run_with_cache(
            base_tokens,
            names_filter=lambda name: name == f'blocks.{layer_l}.hook_resid_post',
            return_type=None,
        )
    return cache[f'blocks.{layer_l}.hook_resid_post'].clone()


def make_run_from_layer(model: HookedTransformer, layer_l: int, t_pos: int,
                         base_resid_at_layer: torch.Tensor
                         ) -> Callable[[torch.Tensor], torch.Tensor]:
    """Return F(h) = final resid_post at t_pos, with h substituted at position t_pos
    into the fixed residual stream at layer_l (all other positions held at their
    natural values from base_resid_at_layer), resuming the forward pass at layer_l via
    start_at_layer.

    Skipping blocks 0..layer_l-1 (they don't depend on h) is what keeps
    jacobian_full_chunked's vmapped Jacobian trace inside the GPU's memory budget --
    with the naive from-scratch-every-call version, jacrev OOM'd even at chunk_size=1.
    """
    def F(h: torch.Tensor) -> torch.Tensor:
        captured = {}
        resid_in = base_resid_at_layer.clone()
        resid_in[:, t_pos, :] = h.to(resid_in.dtype)

        def capture_hook(value, hook):
            captured['out'] = value[:, t_pos, :].squeeze(0)
            return value

        with model.hooks(fwd_hooks=[(FINAL_RESID_KEY, capture_hook)]):
            model(resid_in, start_at_layer=layer_l, return_type=None)

        return captured['out'].float()

    return F


def jacobian_topk(F: Callable[[torch.Tensor], torch.Tensor], h0: torch.Tensor,
                   k: int = 5, n_iter: int = 15, seed: int = 0
                   ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Power-iteration estimate of the top-k singular triples of J = d F / d h at h0.

    Returns
    -------
    sigmas : (k,)      singular values, descending
    V      : (k, d)    right-singular vectors (input space, unit norm)
    U      : (k, d)    left-singular vectors  (output space, unit norm)
    """
    d = h0.numel()
    gen = torch.Generator().manual_seed(seed)
    sigmas, Vs, Us = [], [], []

    for _ in range(k):
        v = torch.randn(d, generator=gen).to(device=h0.device, dtype=h0.dtype)
        v = v / v.norm()
        for prev_v in Vs:
            v = v - (v @ prev_v) * prev_v
        v = v / v.norm()

        u = None
        for _ in range(n_iter):
            _, Jv = torch.func.jvp(F, (h0,), (v,))
            for prev_u in Us:
                Jv = Jv - (Jv @ prev_u) * prev_u
            u = Jv / (Jv.norm() + 1e-12)

            _, JTu = torch.autograd.functional.vjp(F, h0, u, create_graph=False)
            for prev_v in Vs:
                JTu = JTu - (JTu @ prev_v) * prev_v
            v = JTu / (JTu.norm() + 1e-12)

        _, Jv = torch.func.jvp(F, (h0,), (v,))
        sigma = Jv.norm().item()

        sigmas.append(sigma)
        Vs.append(v.detach())
        Us.append(u.detach())

    return torch.tensor(sigmas), torch.stack(Vs), torch.stack(Us)


    """Exact d_model x d_model Jacobian via chunked reverse-mode AD.
    Trades speed for memory -- smaller chunk_size uses less VRAM per pass, more
    sequential passes overall. Use only where the full spectrum is needed (ground-truth
    validation), not as a substitute for jacobian_topk -- this is much more expensive and
    should never be run across all 32 layers.
    """

def jacobian_full_chunked(F: Callable[[torch.Tensor], torch.Tensor], h0: torch.Tensor,
                           chunk_size: int = 64) -> torch.Tensor:
    with torch.no_grad():
        J = torch.func.jacrev(F, chunk_size=chunk_size)(h0)
    return J
    

print('precompute_resid_at_layer / run_from_layer / jacobian_topk / jacobian_full_chunked defined.')

precompute_resid_at_layer / run_from_layer / jacobian_topk / jacobian_full_chunked defined.


## Cell 6 — Gold-standard validation: exact SVD vs. power iteration at `L_STAR`

Formerly Cell 8b of `01e_jacobian_topk_probe.ipynb`. Compares `jacobian_topk`'s
power-iteration estimate against the exact chunked-Jacobian SVD, at the same `h0` loaded
from disk above.

In [6]:
# Fresh kernel means there is nothing stale to clear, but this is cheap insurance if this
# notebook is ever re-run without restarting.
gc.collect()
torch.cuda.empty_cache()

h0_star = mean_h_by_layer[L_STAR].to(DEVICE)

# Precompute once (no grad) -- the fixed residual stream at layer_l for every position,
# which make_run_from_layer resumes from via start_at_layer instead of recomputing
# blocks 0..layer_l-1 (and having jacrev retrace them) on every call to F(h).
base_resid_at_L = precompute_resid_at_layer(model, L_STAR, context_tokens)
F_star = make_run_from_layer(model, L_STAR, T_POS, base_resid_at_L)

# Chunked jacrev still runs `chunk_size` copies of the remaining forward+backward pass
# through vmap simultaneously -- if that overflows VRAM, halve chunk_size and retry
# rather than failing outright. chunk_size=1 is the floor (sequential, slowest, smallest
# footprint); if even that OOMs, the problem isn't the chunk size (this is what happened
# before F_star was switched to start_at_layer -- see the notebook's earlier run history).
chunk_size = JACREV_CHUNK_SIZE
J_full = None
while True:
    try:
        print(f'Computing exact chunked Jacobian at layer {L_STAR} '
              f'(chunk_size={chunk_size}) ...')
        J_full = jacobian_full_chunked(F_star, h0_star, chunk_size=chunk_size)
        break
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if chunk_size == 1:
            raise
        chunk_size = max(1, chunk_size // 2)
        print(f'  OOM -- retrying with chunk_size={chunk_size}')

U_full, S_full, Vt_full = torch.linalg.svd(J_full.float(), full_matrices=False)
sigma1_exact = S_full[0].item()
v1_exact     = Vt_full[0].detach().cpu()

# U_full / S_full / Vt_full are only (d_model, d_model) and (d_model,) -- trivial next to
# J_full's chunked-jacrev working memory. Move them to CPU (freeing their GPU copies)
# instead of deleting them outright: the full-spectrum analysis in the final cell below
# needs all three.
U_full  = U_full.detach().cpu()
S_full  = S_full.detach().cpu()
Vt_full = Vt_full.detach().cpu()

del J_full
gc.collect()
torch.cuda.empty_cache()

# jacobian_topk's estimate at the very same h0 / F, for a like-for-like comparison.
sigma1_poweriter, V_poweriter, _ = jacobian_topk(
    F_star, h0_star, k=1, n_iter=JAC_NITER, seed=JAC_SEED
)
v1_poweriter = V_poweriter[0].detach().cpu()

sigma_rel_err = abs(sigma1_exact - sigma1_poweriter[0].item()) / (sigma1_exact + 1e-12)
cos_v1        = torch.abs(torch.dot(v1_exact, v1_poweriter)).item()

print(f'\n--- Gold-standard validation at layer {L_STAR} ---')
print(f'sigma_1 -- exact SVD:                 {sigma1_exact:.6f}')
print(f'sigma_1 -- power iteration:            {sigma1_poweriter[0].item():.6f}')
print(f'sigma_1 relative error:                {sigma_rel_err:.4e}')
print(f'cos(v1_exact, v1_poweriter):            {cos_v1:.6f}')

gold_standard_result = {
    'layer': L_STAR,
    'sigma1_exact': sigma1_exact,
    'sigma1_poweriter': sigma1_poweriter[0].item(),
    'sigma1_rel_err': sigma_rel_err,
    'cos_v1_exact_vs_poweriter': cos_v1,
    'jac_niter': JAC_NITER,
    'jacrev_chunk_size_used': chunk_size,
}
with open(os.path.join(OUTPUT_ROOT, CATEGORY, 'gold_standard_validation.json'), 'w') as f:
    json.dump(gold_standard_result, f, indent=2)

if sigma_rel_err < 0.05 and cos_v1 > 0.95:
    print('\nPASS: power iteration matches the exact SVD closely -- jacobian_topk is '
          f'trustworthy at layer {L_STAR}.')
else:
    print('\nWARNING: power iteration diverges from the exact SVD at this layer -- '
          'do not trust jacobian_topk results here until this is investigated '
          '(try increasing JAC_NITER).')

print(f'\nSaved: {OUTPUT_ROOT}/{CATEGORY}/gold_standard_validation.json')

Computing exact chunked Jacobian at layer 16 (chunk_size=8) ...

--- Gold-standard validation at layer 16 ---
sigma_1 -- exact SVD:                 70.460564
sigma_1 -- power iteration:            70.434372
sigma_1 relative error:                3.7172e-04
cos(v1_exact, v1_poweriter):            1.000340

PASS: power iteration matches the exact SVD closely -- jacobian_topk is trustworthy at layer 16.

Saved: jacobian_outputs/violence/gold_standard_validation.json


## Cell 7 — Full singular spectrum of `J_full` and alignment with the DIM refusal direction

`J_full = dF/dh` at `L_STAR` describes how a perturbation to the residual stream at that
layer is locally reshaped by the remaining decoder blocks before it reaches the final
layer. Its full SVD (`U_full`, `S_full`, `Vt_full`, kept on CPU by Cell 6 above) is the
ground truth against which `jacobian_topk`'s power iteration is checked -- but it is also
a direct probe of the framework's central hypothesis: that refusal behaviour is governed
by a **low-dimensional** subspace, and that the DIM direction found by mean-difference
(Notebook 01c) picks out a geometrically meaningful direction within it.

Two questions this cell answers:

1. **How low-rank is the local Jacobian, really?** The effective rank (participation
   ratio) and the number of singular values needed to reach 90%/95% of the spectral
   energy give a numeric answer, independent of any particular hypothesis about *which*
   direction matters.
2. **Does the empirically-found DIM refusal direction line up with the directions the
   network is most sensitive to?** Ablating/adding along `v_DIM` only changes model
   behaviour if `v_DIM` has non-trivial overlap with high-singular-value directions of
   `J_full` (`Vt_full`'s rows) -- overlap with a near-zero singular value would mean the
   ablation is largely inert by this layer's local linearisation. Cosine similarity of
   `v_DIM` against `v1_exact` and against the top-k rows of `Vt_full` makes this
   concrete.

In [7]:
print('U_full  shape:', tuple(U_full.shape))
print('S_full  shape:', tuple(S_full.shape))
print('Vt_full shape:', tuple(Vt_full.shape))

# Full singular value spectrum. threshold=float('inf') forces torch to print every entry
# instead of eliding the middle of the (d_model,) vector with '...'.
torch.set_printoptions(threshold=float('inf'), linewidth=120, precision=6)
print(f'\nFull singular value spectrum of J_full at layer {L_STAR} ({S_full.numel()} values):')
print(S_full)
torch.set_printoptions(profile='default')

# --- Effective rank of the local Jacobian -----------------------------------------
# Participation ratio: (sum sigma_i)^2 / sum(sigma_i^2). Equals d_model for a flat
# spectrum (all directions equally sensitive) and 1 for a rank-1 spectrum (a single
# dominant direction) -- a direct numeric read on "how low-dimensional is the local
# refusal-relevant geometry" independent of which direction is examined.
energy = S_full ** 2
total_energy = energy.sum().item()
participation_ratio = (S_full.sum().item() ** 2) / total_energy

cum_energy_frac = torch.cumsum(energy, dim=0) / total_energy
rank_90 = int(torch.searchsorted(cum_energy_frac, 0.90).item()) + 1
rank_95 = int(torch.searchsorted(cum_energy_frac, 0.95).item()) + 1
rank_99 = int(torch.searchsorted(cum_energy_frac, 0.99).item()) + 1

print(f'\n--- Effective rank of J_full at layer {L_STAR} (d_model = {S_full.numel()}) ---')
print(f'sigma_1 / sigma_2 ratio:                {(S_full[0] / S_full[1]).item():.4f}')
print(f'sigma_1 / sigma_last ratio:             {(S_full[0] / S_full[-1]).item():.4e}')
print(f'participation ratio (effective rank):   {participation_ratio:.2f}  '
      f'({100 * participation_ratio / S_full.numel():.2f}% of d_model)')
print(f'# singular values for 90% energy:       {rank_90}')
print(f'# singular values for 95% energy:       {rank_95}')
print(f'# singular values for 99% energy:       {rank_99}')

# --- Alignment of the DIM refusal direction with J_full's sensitivity directions ---
# v_DIM is the mean-difference direction Notebook 01c found for this category at this
# layer (dim_outputs/{CATEGORY}/direction.pt). If ablating/adding along v_DIM actually
# changes model behaviour, v_DIM should overlap with rows of Vt_full that carry
# non-trivial singular value -- not with the flat, near-zero tail of the spectrum.
dim_direction_path = os.path.join('dim_outputs', CATEGORY, 'direction.pt')

if os.path.exists(dim_direction_path):
    v_dim = torch.load(dim_direction_path, map_location='cpu').float().flatten()
    v_dim = v_dim / v_dim.norm()

    cos_vs_v1 = torch.abs(torch.dot(v_dim, v1_exact)).item()

    # Cosine similarity against every right-singular vector, so we can find not just v1
    # but whichever singular direction v_DIM happens to line up with best.
    cos_vs_all = torch.abs(Vt_full @ v_dim)
    best_rank = int(torch.argmax(cos_vs_all).item())
    best_cos = cos_vs_all[best_rank].item()

    # Sensitivity-weighted overlap: how much of v_DIM's unit "mass" sits on singular
    # directions the Jacobian actually amplifies, vs. directions it flattens.
    weighted_alignment = torch.sqrt(torch.sum((cos_vs_all ** 2) * energy) / total_energy).item()

    print(f'\n--- v_DIM ({CATEGORY}, layer {L_STAR}) vs. J_full singular directions ---')
    print(f'cos(v_DIM, v1_exact):                   {cos_vs_v1:.6f}')
    print(f'best-aligned singular direction:        rank {best_rank} '
          f'(sigma = {S_full[best_rank].item():.4f}), cos = {best_cos:.6f}')
    print(f'cos(v_DIM, top-5 singular directions):  '
          f'{[round(c, 4) for c in cos_vs_all[:5].tolist()]}')
    print(f'energy-weighted alignment of v_DIM:      {weighted_alignment:.6f}  '
          '(RMS cosine, weighted by sigma_i^2 -- near 1 means v_DIM sits mostly on '
          'high-sensitivity directions, near 0 means it sits mostly on flat/ignored ones)')

    spectrum_analysis_result = {
        'layer': L_STAR,
        'd_model': S_full.numel(),
        'sigma1_over_sigma2': (S_full[0] / S_full[1]).item(),
        'sigma1_over_sigma_last': (S_full[0] / S_full[-1]).item(),
        'participation_ratio': participation_ratio,
        'num_singular_values_90pct_energy': rank_90,
        'num_singular_values_95pct_energy': rank_95,
        'num_singular_values_99pct_energy': rank_99,
        'cos_v_dim_vs_v1_exact': cos_vs_v1,
        'v_dim_best_aligned_rank': best_rank,
        'v_dim_best_aligned_cos': best_cos,
        'v_dim_energy_weighted_alignment': weighted_alignment,
    }
else:
    print(f'\nNo DIM direction found at {dim_direction_path} -- skipping v_DIM alignment '
          'analysis (spectrum-only results still saved below).')
    spectrum_analysis_result = {
        'layer': L_STAR,
        'd_model': S_full.numel(),
        'sigma1_over_sigma2': (S_full[0] / S_full[1]).item(),
        'sigma1_over_sigma_last': (S_full[0] / S_full[-1]).item(),
        'participation_ratio': participation_ratio,
        'num_singular_values_90pct_energy': rank_90,
        'num_singular_values_95pct_energy': rank_95,
        'num_singular_values_99pct_energy': rank_99,
    }

spectrum_path = os.path.join(OUTPUT_ROOT, CATEGORY, 'jacobian_spectrum_analysis.json')
with open(spectrum_path, 'w') as f:
    json.dump(spectrum_analysis_result, f, indent=2)

print(f'\nSaved: {spectrum_path}')

U_full  shape: (4096, 4096)
S_full  shape: (4096,)
Vt_full shape: (4096, 4096)

Full singular value spectrum of J_full at layer 16 (4096 values):
tensor([7.046056e+01, 3.918137e+01, 3.446905e+01, 2.941969e+01, 2.772956e+01, 2.656883e+01, 2.608252e+01, 2.396629e+01,
        2.343154e+01, 2.206754e+01, 2.184933e+01, 2.124849e+01, 2.033346e+01, 1.973602e+01, 1.933584e+01, 1.885908e+01,
        1.861668e+01, 1.813975e+01, 1.792399e+01, 1.765382e+01, 1.726792e+01, 1.679520e+01, 1.657406e+01, 1.656365e+01,
        1.640181e+01, 1.621523e+01, 1.585560e+01, 1.573402e+01, 1.547200e+01, 1.522591e+01, 1.516111e+01, 1.510450e+01,
        1.498076e+01, 1.489787e+01, 1.461053e+01, 1.460326e+01, 1.457631e+01, 1.415621e+01, 1.413788e+01, 1.407955e+01,
        1.390491e+01, 1.375138e+01, 1.360699e+01, 1.346179e+01, 1.333044e+01, 1.322792e+01, 1.316992e+01, 1.313235e+01,
        1.299682e+01, 1.279755e+01, 1.273015e+01, 1.268255e+01, 1.250670e+01, 1.249579e+01, 1.240639e+01, 1.232201e+01,
        1.2272